# ISIC 2024 - More boosting models and more features

My final model scored 0.1590 on the private part of the Kaggle test set, and the medal solutions are at 0.17 and
above. Two things they did on the metadata side that I didn't:

- a lot more engineered features
- several boosting libraries (LightGBM, XGBoost, CatBoost) and several random seeds, all averaged

This notebook tries both, without touching the CNN. Everything runs on the same 5 patient folds.

1. Wider feature set
2. LightGBM, XGBoost and CatBoost with one seed
3. The same with 5 seeds
4. Averaging the three models and blending with the CNN
5. Is the improvement real?

In [1]:
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')
from isic_pauc import p_auc_tpr
from features import build_features
from features_wide import build_wide_features

DATA_DIR = '../data'
N_FOLDS = 5
SEEDS = [42, 43, 44, 45, 46]

## 1. Wider feature set
`build_features` gives the 57 features from `03_lightgbm` (v3). `build_wide_features` (in `src/features_wide.py`)
adds 24 more ratios and combinations that were used in the public notebooks of the competition, for example
perimeter to area, the total color difference to the surrounding skin, or size times age. It also makes a
patient-normalized version of almost every numeric column instead of only 12 of them.

In [2]:
train = pd.read_csv(f'{DATA_DIR}/train-metadata.csv', low_memory=False)
test = pd.read_csv(f'{DATA_DIR}/test-metadata.csv', low_memory=False)
folds = pd.read_csv(f'{DATA_DIR}/folds.csv')
train = train.merge(folds[['isic_id', 'fold']], on='isic_id')

df_v3, features_v3 = build_features(train, test)
df, features_wide = build_wide_features(train, test)
assert set(features_v3) <= set(features_wide)
print(len(features_v3), 'features in v3')
print(len(features_wide), 'features in the wide set')

57 features in v3
126 features in the wide set


## 2. One function for all three models
Same setup as before: train on 4 folds with all malignant lesions and 1% of the benign ones, predict the 5th
fold. New here:

- `kind` chooses the library. The settings are close to each other (300 to 400 trees, learning rate about 0.05).
  I did not tune them.
- `seeds`: for every seed a new sample of benign lesions is drawn and a new model is trained. The predictions are
  averaged as ranks, because the three libraries don't give probabilities on the same scale.
- CatBoost wants categorical columns as strings, so missing values become the string `'missing'`.

In [3]:
CAT_COLS = [col for col in features_wide if str(df[col].dtype) == 'category']


def make_model(kind, seed):
    if kind == 'lgbm':
        return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.8, random_state=seed, verbose=-1)
    if kind == 'xgb':
        return xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8,
                                 colsample_bytree=0.8, enable_categorical=True, tree_method='hist', random_state=seed)
    if kind == 'cat':
        return CatBoostClassifier(iterations=400, learning_rate=0.06, depth=6, random_seed=seed, verbose=0,
                                  cat_features=CAT_COLS)


def run_cv(kind, features, seeds=(42,)):
    X = df[features]
    if kind == 'cat':
        X = X.copy()
        for col in CAT_COLS:
            X[col] = X[col].astype(object).where(X[col].notna(), 'missing').astype(str)

    oof = np.zeros(len(df))
    for fold in range(N_FOLDS):
        val = (df['fold'] == fold).to_numpy()
        trn = df[~val]
        pred = np.zeros(val.sum())
        for seed in seeds:
            idx = np.concatenate([trn.index[trn['target'] == 1],
                                  trn[trn['target'] == 0].sample(frac=0.01, random_state=seed).index])
            model = make_model(kind, seed)
            model.fit(X.loc[idx], df.loc[idx, 'target'])
            pred += pd.Series(model.predict_proba(X[val])[:, 1]).rank(pct=True).to_numpy() / len(seeds)
        oof[val] = pred
    return oof


def fold_scores(oof):
    return [p_auc_tpr(df.loc[df['fold'] == f, 'target'], oof[(df['fold'] == f).to_numpy()], 0.80) for f in range(N_FOLDS)]


results = {}


def add_result(name, oof):
    scores = fold_scores(oof)
    results[name] = scores
    print(f'{name:38s} {np.mean(scores):.4f} +/- {np.std(scores):.4f}')
    return oof

## 3. One seed: features and libraries
First LightGBM on both feature sets (v3 should give 0.1644 again), then the other two libraries.

In [4]:
for kind in ['lgbm', 'xgb', 'cat']:
    for name, features in [('v3', features_v3), ('wide', features_wide)]:
        add_result(f'{kind} {name}', run_cv(kind, features))

lgbm v3                                0.1644 +/- 0.0079


lgbm wide                              0.1666 +/- 0.0107


xgb v3                                 0.1621 +/- 0.0080


xgb wide                               0.1640 +/- 0.0092


cat v3                                 0.1631 +/- 0.0084


cat wide                               0.1630 +/- 0.0118


- LightGBM v3 gives 0.1644 again, so the function does the same as in `03_lightgbm`.
- The wide features help LightGBM and XGBoost a little (+0.002) and CatBoost not at all. They also make the folds
  less stable (the std goes up for all three).
- With my untuned settings LightGBM is the best single model. XGBoost and CatBoost are 0.001 to 0.003 behind.

## 4. Five seeds
Every model is now the average of 5 models that were trained on different samples of benign lesions. With 1% of
the benign lesions per model, 5 seeds also means the model sees about 5 times as many different benign lesions.

In [5]:
oof = {}
for kind in ['lgbm', 'xgb', 'cat']:
    for name, features in [('v3', features_v3), ('wide', features_wide)]:
        start = time.time()
        oof[f'{kind}_{name}'] = add_result(f'{kind} {name} x5', run_cv(kind, features, seeds=SEEDS))
        print(f'   {time.time() - start:.0f} s')

lgbm v3 x5                             0.1649 +/- 0.0102
   22 s


lgbm wide x5                           0.1669 +/- 0.0108
   37 s


xgb v3 x5                              0.1634 +/- 0.0092
   23 s


xgb wide x5                            0.1660 +/- 0.0099
   39 s


cat v3 x5                              0.1654 +/- 0.0079
   320 s


cat wide x5                            0.1654 +/- 0.0115
   375 s


Averaging 5 seeds changes less than I expected: +0.0005 for LightGBM, +0.001 to +0.002 for the other two. All
six models are now between 0.163 and 0.167. CatBoost is by far the slowest (over 5 minutes for 25 models, the
others need about half a minute).

## 5. Averaging the three libraries and adding the CNN
The CNN predictions are the same out-of-fold predictions as in `05_blend`, and the blend is done the same way:
`w * boosting rank + (1 - w) * CNN rank`. The first line is the final model of the capstone for comparison.

In [6]:
blend_old = pd.read_csv(f'{DATA_DIR}/oof_blend.csv')
assert (blend_old['isic_id'].to_numpy() == df['isic_id'].to_numpy()).all()
cnn_rank = blend_old['cnn_rank'].to_numpy()

add_result('capstone blend (LightGBM v3 + CNN)', blend_old['blend'].to_numpy())
for name in ['v3', 'wide']:
    three = (oof[f'lgbm_{name}'] + oof[f'xgb_{name}'] + oof[f'cat_{name}']) / 3
    oof[f'three_{name}'] = add_result(f'3 libraries {name}', three)
    oof[f'lgbm_cnn_{name}'] = add_result(f'lgbm {name} x5 + CNN (w=0.7)', 0.7 * oof[f'lgbm_{name}'] + 0.3 * cnn_rank)
    for w in [0.6, 0.7, 0.8]:
        oof[f'three_cnn_{name}_{w}'] = add_result(f'3 libraries {name} + CNN (w={w})', w * three + (1 - w) * cnn_rank)

capstone blend (LightGBM v3 + CNN)     0.1691 +/- 0.0067
3 libraries v3                         0.1656 +/- 0.0090
lgbm v3 x5 + CNN (w=0.7)               0.1689 +/- 0.0079


3 libraries v3 + CNN (w=0.6)           0.1684 +/- 0.0068
3 libraries v3 + CNN (w=0.7)           0.1693 +/- 0.0070
3 libraries v3 + CNN (w=0.8)           0.1692 +/- 0.0078


3 libraries wide                       0.1670 +/- 0.0107
lgbm wide x5 + CNN (w=0.7)             0.1706 +/- 0.0084
3 libraries wide + CNN (w=0.6)         0.1695 +/- 0.0077
3 libraries wide + CNN (w=0.7)         0.1705 +/- 0.0083


3 libraries wide + CNN (w=0.8)         0.1705 +/- 0.0092


In [7]:
table = pd.DataFrame(results, index=[f'fold {i}' for i in range(N_FOLDS)]).T
table['mean'] = table.iloc[:, :N_FOLDS].mean(axis=1)
table['std'] = table.iloc[:, :N_FOLDS].std(axis=1, ddof=0)
table.round(4)

,fold 0,fold 1,fold 2,fold 3,fold 4,mean,std
lgbm v3,0.1688,0.1529,0.1689,0.1741,0.1573,0.1644,0.0079
lgbm wide,0.1758,0.1501,0.1712,0.1776,0.1583,0.1666,0.0107
xgb v3,0.1672,0.1526,0.1667,0.1715,0.1525,0.1621,0.0080
xgb wide,0.1720,0.1509,0.1701,0.1720,0.1550,0.1640,0.0092
cat v3,0.1695,0.1517,0.1689,0.1713,0.1540,0.1631,0.0084
cat wide,0.1761,0.1526,0.1666,0.1738,0.1459,0.1630,0.0118
lgbm v3 x5,0.1711,0.1504,0.1690,0.1779,0.1560,0.1649,0.0102
lgbm wide x5,0.1769,0.1519,0.1720,0.1778,0.1560,0.1669,0.0108
xgb v3 x5,0.1700,0.1499,0.1678,0.1739,0.1554,0.1634,0.0092
xgb wide x5,0.1745,0.1528,0.1698,0.1772,0.1558,0.1660,0.0099


- Averaging the three libraries is not better than the best single one: 0.1670 for the three on the wide
  features, 0.1669 for LightGBM alone. They are too similar to each other. Their mistakes are mostly the same
  mistakes, which is the opposite of what happened with the CNN.
- With the CNN added, the best versions reach 0.1705 to 0.1706, compared to 0.1691 for the capstone blend.
- The weight doesn't matter much between 0.7 and 0.8, so I keep 0.7.

## 6. Is the improvement real?
The differences above are in the third and fourth decimal, and the fold-to-fold std is around 0.008. To see if
any of this is more than noise, I use a bootstrap over patients (the same idea as in `09_questions`): draw
patients with replacement, compute the pAUC of the old blend and of the new candidates on the same sample, and
look at the difference.

In [8]:
def bootstrap_diff(new, old, n=300, seed=0):
    rng = np.random.default_rng(seed)
    groups = list(df.groupby('patient_id').indices.values())
    y = df['target'].to_numpy()
    # ranks inside each fold, so that the folds can be pooled
    new = pd.Series(new).groupby(df['fold'].to_numpy()).rank(pct=True).to_numpy()
    old = pd.Series(old).groupby(df['fold'].to_numpy()).rank(pct=True).to_numpy()
    diffs = []
    for _ in range(n):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        diffs.append(p_auc_tpr(y[idx], new[idx], 0.80) - p_auc_tpr(y[idx], old[idx], 0.80))
    return np.array(diffs)


old = blend_old['blend'].to_numpy()
rows = {}
for name, key in [('lgbm v3 x5 + CNN', 'lgbm_cnn_v3'), ('3 libraries v3 + CNN', 'three_cnn_v3_0.7'),
                  ('lgbm wide x5 + CNN', 'lgbm_cnn_wide'), ('3 libraries wide + CNN', 'three_cnn_wide_0.7')]:
    d = bootstrap_diff(oof[key], old)
    lo, hi = np.percentile(d, [2.5, 97.5])
    rows[name] = {'mean difference to the capstone blend': d.mean(), '95% interval': f'{lo:.4f} to {hi:.4f}',
                  'share of samples > 0': (d > 0).mean()}
pd.DataFrame(rows).T

,mean difference to the capstone blend,95% interval,share of samples > 0
lgbm v3 x5 + CNN,-0.000365,-0.0018 to 0.0010,0.31
3 libraries v3 + CNN,0.000107,-0.0015 to 0.0016,0.556667
lgbm wide x5 + CNN,0.001462,-0.0009 to 0.0038,0.853333
3 libraries wide + CNN,0.001365,-0.0008 to 0.0034,0.856667


None of the four candidates is clearly better than the capstone blend. For the two versions with the wide
features the mean difference is about +0.0014 and it is above 0 in about 85% of the samples, but the interval
still includes 0. With the v3 features there is no difference at all.

So on my own data I can't show that all of this work improved the model. The most I can say is that the wide
features *probably* help a little, and that adding XGBoost and CatBoost on top of LightGBM does nothing I can
measure.

In [9]:
pd.DataFrame({'isic_id': df['isic_id'], 'patient_id': df['patient_id'], 'target': df['target'], 'fold': df['fold'],
              'boosting_rank': oof['three_wide'], 'cnn_rank': cnn_rank,
              'blend': oof['three_cnn_wide_0.7']}).to_csv(f'{DATA_DIR}/oof_boosting_blend.csv', index=False)

## 7. Kaggle
The cross-validation can't decide it, so I submitted the candidate with the three libraries, the wide features
and the CNN (w = 0.7) to the hidden test set. The script is `src/kaggle_inference_boosting.py`. The boosting
models are trained inside the Kaggle notebook on all the training data (they only take a few minutes), and the
CNN fold models are the same files as in the first submission.

| Submission | Public | Private |
|---|---|---|
| Capstone blend (LightGBM v3 + CNN) | 0.1706 | 0.1590 |
| 3 libraries, wide features, 5 seeds + CNN | 0.1745 | 0.1604 |

Both parts went up, +0.004 on the public part and +0.0014 on the private part. The private gain is the same size
as the one in my cross-validation (+0.0014), which I couldn't tell apart from noise. So I read it as a small
real improvement at best.

## What I learned
- More boosting libraries were not the missing piece. On this data LightGBM, XGBoost and CatBoost with the same
  features are nearly the same model.
- More features and more seeds gave about +0.001 to +0.002. That is real work for a gain that is smaller than
  the noise of the validation.
- The gap to the medal solutions on the private part is still about 0.010 (0.1604 vs 0.1705). After this
  notebook I'm fairly sure it doesn't come from the boosting side. What's left is the image model: more malignant
  training images and a stronger network.